# EDA 2 — Golden Benchmark (baseline de legibilidade e comandos)

**Épico 4.2 (#38) · RF-10, RNF-04, RNF-05, RNF-06**

Objetivo: mapear a distribuição de **baseline** do Flesch PT-BR e da **densidade de comandos conjugados** nas questões do Golden Benchmark (30 a 50 questões escolares), antes de qualquer adaptação pela Íris.

> ⚠️ Se o dataset estiver com `status = "rascunho"`, ele foi escrito para validar o pipeline e **ainda não passou por revisão pedagógica**. Os números só valem como baseline depois que o status passar para `"validado"`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Localiza a raiz do projeto (pasta com pyproject.toml) e expõe src/ para import.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from iris.eval.readability import analyze_question

DATA = ROOT / "data" / "golden_benchmark.json"
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Raiz do projeto:", ROOT)

## 1. Carga e visão geral do dataset

In [ ]:
questions = json.loads(DATA.read_text(encoding="utf-8"))
if not isinstance(questions, list):
    raise ValueError("O JSON do benchmark deve ser uma lista de questões.")

rows = []
for i, q in enumerate(questions):
    if "enunciado" not in q:
        raise ValueError(f"Questão {i} sem o campo 'enunciado'.")
    qid = str(q.get("id", f"Q{i + 1:02d}"))
    stats = analyze_question(qid, q["enunciado"]).__dict__
    meta = {k: v for k, v in q.items() if k != "enunciado"}
    rows.append({**meta, **stats, "id": qid})
df = pd.DataFrame(rows)

COLUNAS = ["flesch", "flesch_band", "n_commands", "commands_per_100_words", "has_conjugated_command", "n_words"]
faltando = [c for c in COLUNAS if c not in df.columns]
assert not faltando, f"analyze_question não devolveu as colunas esperadas: {faltando}"

status = df["status"].value_counts().to_dict() if "status" in df else "sem campo 'status'"
print(f"{len(df)} questões | status: {status}")
if not 30 <= len(df) <= 50:
    print("⚠️ A meta do benchmark é de 30 a 50 questões.")

ordem_ano = sorted(df["ano"].unique()) if "ano" in df else []
display(pd.crosstab(df["disciplina"], df["ano"])[ordem_ano] if ordem_ano else df.head())

## 2. Flesch PT-BR

Fórmula (Martins et al.): `248,835 − 1,015 × (palavras/sentenças) − 84,6 × (sílabas/palavras)`.
Quanto maior o índice, mais fácil o texto. Faixas: ≥ 75 muito fácil · 50–74 fácil · 25–49 difícil · < 25 muito difícil.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df["flesch"].dropna(), bins=10, color="#4C78A8", edgecolor="white")
axes[0].axvline(df["flesch"].mean(), color="#E45756", ls="--", label=f"média = {df['flesch'].mean():.1f}")
axes[0].axvline(df["flesch"].median(), color="#54A24B", ls=":", label=f"mediana = {df['flesch'].median():.1f}")
axes[0].set(title="Distribuição do Flesch PT-BR", xlabel="Índice de Flesch", ylabel="Nº de questões")
axes[0].legend()

disc = sorted(df["disciplina"].unique())
axes[1].boxplot([df.loc[df["disciplina"] == d, "flesch"].dropna() for d in disc], tick_labels=disc)
axes[1].set(title="Flesch por disciplina", ylabel="Índice de Flesch")

plt.tight_layout()
plt.savefig(REPORTS / "eda2_flesch.png")
plt.show()

In [ ]:
faixas = ["muito fácil", "fácil", "difícil", "muito difícil"]
contagem = df["flesch_band"].value_counts().reindex(faixas, fill_value=0)

ax = contagem.plot.bar(color="#4C78A8", rot=0, figsize=(6, 3.5))
ax.set(title="Questões por faixa de dificuldade", xlabel="", ylabel="Nº de questões")
plt.tight_layout()
plt.savefig(REPORTS / "eda2_flesch_faixas.png")
plt.show()

if ordem_ano:
    display(
        df.groupby("ano")["flesch"].agg(["count", "mean"]).round(1)
        .rename(columns={"count": "n", "mean": "Flesch médio"}).loc[ordem_ano]
    )

**Como ler:** enunciados curtos (ex.: "Cite os pontos cardeais.") oscilam muito no Flesch, porque poucas palavras e uma única sentença dominam a média. Interprete a **distribuição**, não uma questão isolada.

## 3. Densidade de comandos conjugados

Um comando é **conjugado** quando divide a mesma sentença com outro comando (ex.: *"Leia o texto e responda"*). Esse é o padrão que tende a sobrecarregar a memória de trabalho de alunos com TDAH.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df["n_commands"].value_counts().sort_index().plot.bar(ax=axes[0], color="#F58518", rot=0)
axes[0].set(title="Comandos por questão", xlabel="Nº de comandos", ylabel="Nº de questões")

pct = df.groupby("disciplina")["has_conjugated_command"].mean().mul(100).sort_values()
pct.plot.barh(ax=axes[1], color="#B279A2")
axes[1].set(title="% de questões com comando conjugado", xlabel="%", ylabel="")
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.savefig(REPORTS / "eda2_comandos.png")
plt.show()

## 4. Validação do detector de comandos

In [ ]:
if "comando_conjugado_esperado" in df.columns:
    esperado = df["comando_conjugado_esperado"].astype(bool)
    detectado = df["has_conjugated_command"].astype(bool)
    print(f"Concordância detector × rótulo manual: {(esperado == detectado).mean():.0%} ({(esperado == detectado).sum()}/{len(df)})")
    divergentes = df.loc[esperado != detectado, ["id", "disciplina", "n_commands", "n_conjugated_commands"]]
    display(divergentes) if len(divergentes) else print("Nenhuma divergência.")
else:
    print("Dataset sem rótulo manual: validação do detector não executada.")

## 5. Baseline consolidado

In [ ]:
flesch = df["flesch"].dropna()
baseline = {
    "n_questoes": int(len(df)),
    "flesch_media": round(float(flesch.mean()), 2),
    "flesch_mediana": round(float(flesch.median()), 2),
    "flesch_desvio": round(float(flesch.std()), 2) if len(flesch) > 1 else None,
    "flesch_faixas": df["flesch_band"].value_counts().to_dict(),
    "comandos_por_questao_media": round(float(df["n_commands"].mean()), 2),
    "comandos_por_100_palavras_media": round(float(df["commands_per_100_words"].mean()), 2),
    "pct_questoes_com_comando_conjugado": round(100 * float(df["has_conjugated_command"].mean()), 1),
    "status_dataset": df["status"].value_counts().to_dict() if "status" in df else None,
}

(REPORTS / "eda2_baseline.json").write_text(json.dumps(baseline, ensure_ascii=False, indent=2), encoding="utf-8")
df.to_csv(REPORTS / "eda2_questoes.csv", index=False)

print(json.dumps(baseline, ensure_ascii=False, indent=2))
print("\nArquivos gerados em", REPORTS)

## 6. Conclusões e limitações

- Este baseline é o **ponto de comparação** para a Semana 3: o Flesch das saídas da Íris deve ficar acima do baseline (texto mais simples) e a densidade de comandos conjugados deve cair, sem perda de conteúdo curricular (TRCC ≥ 95%).
- A silabação usa hifenização (`pyphen`), que aproxima a divisão silábica: pode errar em ditongos e hiatos. Serve para comparar antes × depois, mas não como medida absoluta.
- A lista de verbos de comando (`COMMAND_VERBS`) é um ponto de partida; revise-a quando o dataset for validado pela equipe pedagógica.
- Questões **sem verbo de comando** (ex.: problemas com pergunta direta) contam 0 comandos; isso é esperado, não erro.
- Este notebook usa só `analyze_question`, então não depende do formato de `summarize`/`analyze_benchmark`, que podem variar entre versões do `readability.py`.